In [ ]:
import pandas as pd

customers = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    "../data/raw/olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    "../data/raw/olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    "../data/raw/olist_products_dataset.csv"
)

sellers = pd.read_csv(
    "../data/raw/olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    "../data/raw/product_category_name_translation.csv"
)

products.head()

In [ ]:
#customers.shape
#customers.columns
#customers.info()
#customers.describe(include="all")



In [12]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(
        f"{name:25} "
        f"Rows: {df.shape[0]:>8,} | "
        f"Columns: {df.shape[1]:>3}"
    )

customers                 Rows:   99,441 | Columns:   5
orders                    Rows:   99,441 | Columns:   8
order_items               Rows:  112,650 | Columns:   7
payments                  Rows:  103,886 | Columns:   5
reviews                   Rows:   99,224 | Columns:   7
products                  Rows:   32,951 | Columns:   9
sellers                   Rows:    3,095 | Columns:   4
category_translation      Rows:       71 | Columns:   2


In [14]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("Duplicate rows:", df.duplicated().sum())


customers
Duplicate rows: 0

orders
Duplicate rows: 0

order_items
Duplicate rows: 0

payments
Duplicate rows: 0

reviews
Duplicate rows: 0

products
Duplicate rows: 0

sellers
Duplicate rows: 0

category_translation
Duplicate rows: 0


In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isna().sum())

In [16]:
def inspect_dataframe(df, name):
    print("=" * 60)
    print(f"TABLE: {name}")
    print("=" * 60)

    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isna().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

In [ ]:
for name, df in datasets.items():
    inspect_dataframe(df, name)

In [ ]:
customer_counts = (
    customers
    .groupby("customer_unique_id") #groups records by customer.
    .size() #counts rows in each group.
    .sort_values(ascending=False) #puts the largest counts first.
)

# How many order-linked customer records belong to each unique customer?

customer_counts.head(10)

In [ ]:
#Check orders
print("Rows:", len(orders))

print(
    "Unique order_id:",
    orders["order_id"].nunique() # nunique() - It counts distinct values.
)

print(
    "Duplicate order_id:",
    orders["order_id"].duplicated().sum()
)

In [ ]:
# Check order_items
print("Rows:", len(order_items))

print(
    "Unique orders:",
    order_items["order_id"].nunique()
)

print(
    "Unique products:",
    order_items["product_id"].nunique()
)

In [ ]:
# Calculate Items Per Order
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
)

items_per_order.describe()

# if the mean is around 1.14:
# Most orders contain approximately one item, but some contain multiple items.

In [26]:
# Check payments
print("Payment rows:", len(payments))

print(
    "Unique orders with payment:",
    payments["order_id"].nunique()
)

print(
    "Payment types:",
    payments["payment_type"].unique()
)

payments["payment_type"].value_counts()

Payment rows: 103886
Unique orders with payment: 99440
Payment types: <StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [ ]:
payment_counts = (
    payments
    .groupby("order_id")
    .size()
)

payment_counts.value_counts().sort_index()

Important: Payment Grain
Payment data is not necessarily one row per order.
An order can have multiple payment records.
Check:
payment_counts = (
    payments
    .groupby("order_id")
    .size()
)

payment_counts.value_counts().sort_index()

You might see something like:
1 payment → many orders
2 payments → some orders
3 payments → fewer
This matters enormously.
If you join orders to payments and then calculate:
SUM(order_value)
you could potentially duplicate order-level values.
Again:
Always understand grain before joining.

In [ ]:
# check reviews
print("Review rows:", len(reviews))

print(
    "Unique orders reviewed:",
    reviews["order_id"].nunique()
)

print(
    "Review scores:",
    sorted(reviews["review_score"].dropna().unique())
)

reviews["review_score"].value_counts().sort_index()

#This will tell us how customer ratings are distributed.

In [ ]:
# Check Products
print("Product rows:", len(products))

print(
    "Unique product_id:",
    products["product_id"].nunique()
)

print(
    "Duplicate product_id:",
    products["product_id"].duplicated().sum()
)

products["product_category_name"].nunique()

#This tells us how many distinct product categories exist in the raw dataset.

In [ ]:
# Check Sellers
print("Seller rows:", len(sellers))

print(
    "Unique seller_id:",
    sellers["seller_id"].nunique()
)

print(
    "Duplicate seller_id:",
    sellers["seller_id"].duplicated().sum()
)



In [34]:
# Missing Values — Properly
def missing_summary(df):
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        )
    })

    return (
        result
        .sort_values(
            "missing_percentage",
            ascending=False
        )
    )

missing_summary(orders)

,missing_count,missing_percentage
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899
order_id,0,0.000000
order_purchase_timestamp,0,0.000000
order_status,0,0.000000
customer_id,0,0.000000
order_estimated_delivery_date,0,0.000000


In [35]:
# Investigate order staus
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [ ]:
# compare missing delivery dates against order status.
pd.crosstab(
    orders["order_status"],
    orders["order_delivered_customer_date"].isna()
)

#   Delivery Date is missing (True) or present (False)

order_delivered_customer_date,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


2. The Core Insights (What the table shows)
Looking at the numbers, you can instantly see a logical pattern:
• Why are most dates missing? (The True Column)
The vast majority of missing dates belong to orders that are shipped (1,107), canceled (619), unavailable (609), or invoiced/processing. This makes perfect sense: if an order was canceled, processing, or still on the delivery truck, it shouldn't have a delivery date yet!
• The Normal Behavior (The False Column)
There are 96,470 orders that are marked as delivered and successfully have a delivery date (False for missing). This is exactly what you want to see.
3. Spotting the Data Anomalies (The Red Flags)
This table is incredibly useful because it also exposes minor errors or edge cases in your data that you need to fix:
• The 8 Missing Deliveries: There are 8 orders marked as delivered, but their delivery date is True (missing). This represents a minor data tracking error.
• The 6 Ghost Deliveries: There are 6 orders marked as canceled, yet they somehow have a delivery date (False).


In [39]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce" #If Python encounters an invalid date, instead of crashing: invalid date → NaT (Not a Time) 
    )

#converts text such as:
#       2017-10-02 10:56:33
#       into a real datetime value. 

In [40]:
# Check Date Ranges
print(
    "Earliest order:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Latest order:",
    orders["order_purchase_timestamp"].max()
)

Earliest order: 2016-09-04 21:15:19
Latest order: 2018-10-17 17:30:18


In [41]:
# Check for Impossible Dates, Delivery shouldn't normally happen before purchase.
invalid_delivery = orders[
    orders["order_delivered_customer_date"]
    < orders["order_purchase_timestamp"]
]

len(invalid_delivery)

0

In [ ]:
# Delivery Delay, Actual delivery date - Estimated delivery date = Delivery delay

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60) #It divides the total seconds by the number of seconds in a single day (24 hours × 60 minutes × 60 seconds). This converts the final value into standard days.

#len(orders[orders["delivery_delay_days"] > 0])
orders["is_late"] = (
    orders["delivery_delay_days"] > 0
)
orders["is_late"].value_counts(dropna=False)

is_late
False    91614
True      7827
Name: count, dtype: int64

In [51]:
# Check Price and Freight Values
order_items[
    ["price", "freight_value"]
].describe()


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [56]:
order_items[
    ["price", "freight_value"]
].isna().sum()

(order_items["price"] < 0).sum()

(order_items["freight_value"] < 0).sum()

np.int64(0)

In [57]:
# Find extreme prices 
order_items.nlargest(
    10,
    "price"
)[
    ["order_id", "product_id", "price", "freight_value"]
]

,order_id,product_id,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,489ae2aa008f021502940f251d4cce7f,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,69c590f7ffc7bf8db97190b6cb6ed62e,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1bdf5e6731585cf01aa8169c7028d6ad,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,a6492cc69376c469ab6f61d8f44de961,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,c3ed642d592594bb648ff4a04cee2747,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,259037a6a41845e455183f89c5035f18,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,a1beef8f3992dbd4cd8726796aa69c53,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,6cdf8fc1d741c76586d8b6b15e9eef30,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,dd113cb02b2af9c8e5787e8f1f0722f6,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,6902c1962dd19d540807d0ab8fade5c6,3999.90,17.01


This gives the 10 highest-priced items.
Why do this?
Not because we're automatically deleting them.
An extreme value could be:
- legitimate luxury/high-value product,
- data entry problem,
- unusual transaction.
Outlier ≠ error.
This distinction matters.

In [60]:
# Relationship Validation
# Now we begin checking whether our foreign keys actually work.
# Does every order have a customer?

missing_customer_orders = orders[
    ~orders["customer_id"].isin(  # isin() - Does this order's customer ID exist in the customer table?
        customers["customer_id"]  # The ~ reverses the condition: Give me IDs that do not exist.
    )
]

len(missing_customer_orders)

0

In [61]:
#Check Order Items → Orders
missing_item_orders = order_items[
    ~order_items["order_id"].isin(
        orders["order_id"]
    )
]

len(missing_item_orders)

0

In [ ]:
# Check Products : Does every product appearing in an order actually exist in the product master table?
missing_products = order_items[
    ~order_items["product_id"].isin(
        products["product_id"]
    )
]

len(missing_products)

0

In [63]:
# Check Sellers
missing_sellers = order_items[
    ~order_items["seller_id"].isin(
        sellers["seller_id"]
    )
]

len(missing_sellers)

0

In [66]:
# Build a Data Quality Summary
# Let's automate the checks.

quality_summary = []

for name, df in datasets.items():

    quality_summary.append({
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum()
    })

quality_summary = pd.DataFrame(quality_summary)

quality_summary

,table,rows,columns,duplicate_rows,missing_cells
0,customers,99441,5,0,0
1,orders,99441,10,0,7873
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,0,145903
5,products,32951,9,0,2448
6,sellers,3095,4,0,0
7,category_translation,71,2,0,0
